In [1]:
from langchain_neo4j import Neo4jGraph
from configuration.config import *

graph = Neo4jGraph(
    url=NEO4J_CONFIG["uri"],
    username=NEO4J_CONFIG["auth"][0],
    password=NEO4J_CONFIG["auth"][1],
)

In [2]:
print(graph.schema)

Node properties:
Category1 {name: STRING, id: INTEGER}
Category2 {name: STRING, id: INTEGER}
Category3 {name: STRING, id: INTEGER}
BaseAttrName {name: STRING, id: INTEGER}
BaseAttrValue {name: STRING, id: INTEGER}
SPU {name: STRING, id: INTEGER}
SKU {name: STRING, id: INTEGER}
Trademark {name: STRING, id: INTEGER}
SaleAttrName {name: STRING, id: INTEGER}
SaleAttrValue {name: STRING, id: INTEGER}
Tag {name: STRING, id: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:BaseAttrName)-[:Have]->(:BaseAttrValue)
(:SPU)-[:Have]->(:Tag)
(:SPU)-[:Have]->(:SaleAttrName)
(:SPU)-[:Belong]->(:Trademark)
(:SPU)-[:Belong]->(:Category3)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:SaleAttrName)-[:Have]->(:SaleAttrValue)


In [3]:
# 查询
cypher = "MATCH (n) RETURN n LIMIT 5"
graph.query(cypher)

[{'n': {'name': '图书、音像、电子书刊', 'id': 1}},
 {'n': {'name': '手机', 'id': 2}},
 {'n': {'name': '家用电器', 'id': 3}},
 {'n': {'name': '数码', 'id': 4}},
 {'n': {'name': '家居家装', 'id': 5}}]

In [4]:
from langchain_deepseek import ChatDeepSeek

# 定义大模型
llm = ChatDeepSeek(model='deepseek-chat', temperature=0, api_key=API_KEY)

In [5]:
from langchain_neo4j import GraphCypherQAChain

# 定义chain
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)

In [6]:
result = chain.invoke({"query": "苹果有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '苹果'})<-[:Belong]-(spu:SPU)
RETURN spu.name AS product_name, spu.id AS product_id
Full Context:
[{'product_name': 'Apple iPhone 12', 'product_id': 3}, {'product_name': 'Apple iPhone 16 Pro', 'product_id': 19}]

> Finished chain.


In [7]:
result = chain.invoke({"query": "Apple有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (t:Trademark {name: 'Apple'})<-[:Belong]-(spu:SPU)
RETURN spu.name AS product_name, spu.id AS product_id

Full Context:
[]

> Finished chain.
